# Compare years

Measure per-pixel embedding change between the selected years.

`VIEW` chooses the comparison: `pair` measures change between exactly two `YEARS`; `churn` adds changes between consecutive available observations; `when` shows the later year of the largest change. Pixels need at least two available years. Archive zones are derived from the AOI during loading.

### Install dependencies

In [ ]:
%pip install 'numpy>=2' 'zarr>=3' obstore matplotlib pyproj rasterio shapely


### Inputs

In [ ]:
# AOI coordinates are [longitude, latitude].
AOI = [[[109.14468300877257,19.643497070648664],[109.22936355852482,19.643497070648664],[109.22936355852482,19.692688865652542],[109.14468300877257,19.692688865652542],[109.14468300877257,19.643497070648664]]]
YEARS = [2021,2025]
VIEW = "pair"
EMBEDDING_DIMS = 64 # Use 128 for the full embedding
STORE_URL = "s3://olmoearth/olmoearth-v1-3-embeddings/embeddings.zarr"
STORE_ENDPOINT = "https://data.source.coop"


### Archive and GIS functions

In [ ]:
from dataclasses import dataclass
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import zarr
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import FuncFormatter
from obstore.store import S3Store
from pyproj import CRS, Transformer
from rasterio.features import geometry_mask
from rasterio.transform import Affine
from rasterio.windows import from_bounds
from shapely.geometry import Polygon, box, mapping
from shapely.ops import transform as transform_geometry

@dataclass
class AoiEmbeddings:
    zone: int
    embeddings: np.ndarray
    inside: np.ndarray
    bounds: tuple[int, int, int, int]


@dataclass
class ZoneComparisonResult:
    aoi: AoiEmbeddings
    scores: np.ndarray


EMBEDDINGS_ARRAY = "embeddings"
NODATA = -128
CHUNK_SIZE = 256
GRID_WIDTH = 98304
GRID_HEIGHT = 1867776
GRID_TRANSFORM = Affine(10, 0, 0, 0, -10, 9502720)
CHANGE_SCALE_MAX = 0.5


def open_archive(store_url: str, endpoint: str) -> Any:
    store = S3Store.from_url(store_url, endpoint=endpoint, skip_signature=True)
    return zarr.open_group(zarr.storage.ObjectStore(store, read_only=True), mode="r")



LONGITUDE_ORIGIN = -180
UTM_ZONE_WIDTH = 6


def utm_zone(lon: float) -> int:
    return min(60, int((lon - LONGITUDE_ORIGIN) % 360 // UTM_ZONE_WIDTH) + 1)


def zone_group(zone: int) -> str:
    return f"utm{zone:02d}"


def zones_for_aoi(rings: list[list[list[float]]]) -> list[int]:
    longitudes = [lon for ring in rings for lon, _ in ring]
    if not longitudes:
        return []
    first = utm_zone(min(longitudes))
    last = utm_zone(max(longitudes))
    if first <= last:
        return list(range(first, last + 1))
    return list(range(first, 61)) + list(range(1, last + 1))


def zone_band(zone: int, aoi_west: float, aoi_east: float) -> tuple[float, float]:
    west = LONGITUDE_ORIGIN + (zone - 1) * UTM_ZONE_WIDTH
    middle = (aoi_west + aoi_east) / 2
    west += round((middle - west - UTM_ZONE_WIDTH / 2) / 360) * 360
    return west, west + UTM_ZONE_WIDTH


SOUTHERN_LATITUDE_LIMIT = -90
NORTHERN_LATITUDE_LIMIT = 90


def zone_longitude_bounds(
    zone: int, aoi_west: float, aoi_east: float, zones: list[int]
) -> tuple[float, float]:
    west, east = zone_band(zone, aoi_west, aoi_east)
    if zone == zones[0]:
        west = min(west, aoi_west)
    if zone == zones[-1]:
        east = max(east, aoi_east)
    return west, east


def aoi_geometry(
    rings: list[list[list[float]]], zone: int, zones: list[int]
) -> tuple[Any, Transformer]:
    crs = CRS.from_proj4(f"+proj=utm +zone={zone} +datum=WGS84 +units=m +no_defs")
    to_utm = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    polygon = Polygon(rings[0])
    for ring in rings[1:]:
        polygon = polygon.symmetric_difference(Polygon(ring))
    west, east = zone_longitude_bounds(zone, polygon.bounds[0], polygon.bounds[2], zones)
    clipped = polygon.intersection(
        box(west, SOUTHERN_LATITUDE_LIMIT, east, NORTHERN_LATITUDE_LIMIT)
    )
    if clipped.is_empty or clipped.area == 0:
        return None, to_utm
    return transform_geometry(to_utm.transform, clipped), to_utm


def chunk_bounds(window: Any) -> tuple[int, int, int, int]:
    x0 = max(0, int(np.floor(window.col_off / CHUNK_SIZE) * CHUNK_SIZE))
    y0 = max(0, int(np.floor(window.row_off / CHUNK_SIZE) * CHUNK_SIZE))
    x1 = min(GRID_WIDTH, int(np.ceil((window.col_off + window.width) / CHUNK_SIZE) * CHUNK_SIZE))
    y1 = min(GRID_HEIGHT, int(np.ceil((window.row_off + window.height) / CHUNK_SIZE) * CHUNK_SIZE))
    return x0, y0, x1, y1


def year_index_for(available: np.ndarray, year: int) -> int:
    matches = np.flatnonzero(available == year)
    if not len(matches):
        raise ValueError(f"Year {year} is unavailable. Available years: {available.tolist()}")
    return int(matches[0])


def load_embeddings_for_aoi(
    archive: Any, rings: list[list[list[float]]], years: list[int], dims: int
) -> list[AoiEmbeddings]:
    results = []
    zones = zones_for_aoi(rings)
    for zone in zones:
        geometry, _ = aoi_geometry(rings, zone, zones)
        if geometry is None:
            continue
        window = from_bounds(*geometry.bounds, transform=GRID_TRANSFORM)
        x0, y0, x1, y1 = chunk_bounds(window)
        if x1 <= x0 or y1 <= y0:
            continue
        group = archive[zone_group(zone)]
        available = np.asarray(group["time"][:]).astype(int)
        array = group[EMBEDDINGS_ARRAY]
        embeddings = np.stack([
            np.asarray(array[year_index_for(available, year), :dims, y0:y1, x0:x1], dtype=np.int8)
            for year in years
        ])
        transform = GRID_TRANSFORM * Affine.translation(x0, y0)
        inside = geometry_mask(
            [mapping(geometry)], out_shape=(y1 - y0, x1 - x0), transform=transform, invert=True
        )
        results.append(AoiEmbeddings(zone, embeddings, inside, (x0, y0, x1, y1)))
    if not results:
        raise ValueError("The AOI does not overlap any UTM zone.")
    return results


### Analysis functions

In [ ]:
def unit_vectors(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=0)
    return vectors / np.maximum(norms, 1e-12)[None, :, :]


def dequantize(values: np.ndarray) -> np.ndarray:
    scaled = values.astype(np.float32) / 127.5
    return np.sign(scaled) * np.abs(scaled) ** 2


PARITY_SAMPLE_CELLS = 16384
PARITY_MIN_PIXELS = 64


# The archive's encoder leaves a fixed offset per (row % 2, column % 2) grid position,
# which draws one-pixel stripes. Its size varies by region, so it is estimated per area.
def parity_offsets(embeddings: np.ndarray, x0: int, y0: int) -> np.ndarray:
    bands, height, width = embeddings.shape
    cells_down, cells_across = height // 2, width // 2
    step = max(1, int(np.ceil(np.sqrt(cells_down * cells_across / PARITY_SAMPLE_CELLS))))
    means = np.zeros((2, 2, bands), dtype=np.float32)
    for dy in (0, 1):
        for dx in (0, 1):
            sample = embeddings[:, dy:cells_down * 2:step * 2, dx:cells_across * 2:step * 2]
            sample = sample.reshape(bands, -1)
            sample = sample[:, sample[0] != NODATA]
            if sample.shape[1] < PARITY_MIN_PIXELS:
                return np.zeros((2, 2, bands), dtype=np.float32)
            means[(y0 + dy) % 2, (x0 + dx) % 2] = dequantize(sample).mean(axis=1)
    return means - means.mean(axis=(0, 1))


def corrected_vectors(embeddings: np.ndarray, x0: int, y0: int) -> np.ndarray:
    offsets = parity_offsets(embeddings, x0, y0)
    vectors = dequantize(embeddings)
    for dy in (0, 1):
        for dx in (0, 1):
            vectors[:, dy::2, dx::2] -= offsets[(y0 + dy) % 2, (x0 + dx) % 2][:, None, None]
    return vectors


def score_years(embeddings: np.ndarray, inside: np.ndarray, x0: int, y0: int) -> np.ndarray:
    if VIEW == "pair" and len(YEARS) != 2:
        raise ValueError("Pair comparison requires exactly two years.")
    if VIEW not in ("pair", "churn", "when"):
        raise ValueError(f"Unknown comparison view: {VIEW}")
    scores = np.full(inside.shape, np.nan, dtype=np.float32)
    previous = np.zeros(embeddings.shape[1:], dtype=np.float32)
    previous_valid = np.zeros(inside.shape, dtype=bool)
    count = np.zeros(inside.shape, dtype=np.uint8)
    best_jump = np.full(inside.shape, -np.inf, dtype=np.float32)

    for index, values in enumerate(embeddings):
        valid = inside & ~np.all(values == NODATA, axis=0)
        current = unit_vectors(corrected_vectors(values, x0, y0))
        both = valid & previous_valid
        if np.any(both):
            jump = 1 - np.sum(previous * current, axis=0)
            if VIEW == "pair":
                scores[both] = jump[both]
            elif VIEW == "churn":
                scores[both & (count == 1)] = 0
                scores[both] += jump[both]
            else:
                better = both & (jump > best_jump)
                scores[better] = index
                best_jump[better] = jump[better]
        previous[:, valid] = current[:, valid]
        previous_valid |= valid
        count[valid] += 1
    return scores


def compare_years(data: list[AoiEmbeddings]) -> list[ZoneComparisonResult]:
    results = [
        ZoneComparisonResult(part, score_years(part.embeddings, part.inside, *part.bounds[:2]))
        for part in data
    ]
    if not any(np.isfinite(result.scores).any() for result in results):
        raise ValueError("No pixel in the AOI has data for at least two selected years.")
    return results


### Plotting functions

In [ ]:
def crop_to_aoi(
    scores: np.ndarray, inside: np.ndarray, bounds: tuple[int, int, int, int]
) -> tuple[np.ndarray, tuple[float, float, float, float]]:
    x0, y0, x1, y1 = bounds
    rows, columns = np.where(inside)
    if len(rows):
        row_start, row_stop = int(rows.min()), int(rows.max()) + 1
        col_start, col_stop = int(columns.min()), int(columns.max()) + 1
        scores = scores[row_start:row_stop, col_start:col_stop]
        x0, y0 = x0 + col_start, y0 + row_start
        x1, y1 = x0 + col_stop - col_start, y0 + row_stop - row_start
    left, top = GRID_TRANSFORM * (x0, y0)
    right, bottom = GRID_TRANSFORM * (x1, y1)
    return scores, (left, right, bottom, top)


def plot_results(results: list[ZoneComparisonResult]) -> None:
    fig, axes = plt.subplots(
        1, len(results), figsize=(6 * len(results), 6), squeeze=False, layout="constrained"
    )
    valid_scores = np.concatenate([
        result.scores[np.isfinite(result.scores)] for result in results
    ])
    if VIEW == "when":
        cmap = ListedColormap(plt.get_cmap("tab20")(np.linspace(0, 1, len(YEARS))))
        cmap.set_bad("#e5e7eb")
        norm = BoundaryNorm(np.arange(len(YEARS) + 1) - 0.5, len(YEARS))
    else:
        cmap = "YlOrRd"
        norm = None
    color_min = 0 if VIEW == "pair" else float(np.min(valid_scores))
    color_max = CHANGE_SCALE_MAX if VIEW == "pair" else float(np.max(valid_scores))
    if color_max <= color_min:
        color_max = color_min + 1e-6
    for axis, result in zip(axes[0], results):
        display, extent = crop_to_aoi(
            result.scores, result.aoi.inside, result.aoi.bounds
        )
        image = axis.imshow(
            display, cmap=cmap, norm=norm,
            vmin=None if VIEW == "when" else color_min,
            vmax=None if VIEW == "when" else color_max,
            extent=extent, interpolation="nearest", aspect="equal",
        )
        axis.set_title(f"UTM zone {result.aoi.zone}")
        axis.set_xlabel("UTM easting (km)")
        axis.set_ylabel("UTM northing (km)")
        axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:g}"))
        axis.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:g}"))
        axis.set_facecolor("#e5e7eb")
    if VIEW == "when":
        bar = fig.colorbar(image, ax=list(axes[0]), ticks=np.arange(1, len(YEARS)), label="Year of largest change")
        bar.ax.set_yticklabels([str(year) for year in YEARS[1:]])
    else:
        label = "Change (1 − cosine)" if VIEW == "pair" else "Total change (Σ 1 − cosine)"
        fig.colorbar(image, ax=list(axes[0]), label=label)
    title = f"{YEARS[0]} → {YEARS[-1]}" if VIEW == "pair" else f"{YEARS[0]}–{YEARS[-1]}"
    fig.suptitle(f"{VIEW.capitalize()} comparison · {title}")
    plt.show()


### Load embeddings

In [ ]:
print("Loading embeddings...")
archive = open_archive(STORE_URL, STORE_ENDPOINT)
data = load_embeddings_for_aoi(archive, AOI, YEARS, EMBEDDING_DIMS)
print("Embeddings loaded.")


### Run analysis

In [ ]:
print("Running analysis...")
results = compare_years(data)
print("Analysis complete.")


### Visualize results

In [ ]:
plot_results(results)
